# Capitolo 15 – Ricorrenze: RecursiveSum, Torre di Hanoi e forme chiuse (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap15_ricorrenze_java.ipynb)

Verifica numerica degli esempi del capitolo: l'algoritmo **RecursiveSum** con il conteggio dei passi $S(n)$ (che soddisfa $S(1)=2$, $S(n)=S(n-1)+3$, cioè $S(n)=3n-1$), la **Torre di Hanoi** che conta le mosse ($2^n-1$) e il confronto fra alcune ricorrenze del capitolo, calcolate direttamente, e la loro forma chiusa.

Il capitolo completo, con pseudocodice, dimostrazioni ed esempi, è [sul sito](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/15-ricorrenze/).

Ogni programma è scritto in un file `.java` con la cella `%%writefile` e poi compilato ed eseguito con `javac` e `java`: si eseguono le celle in ordine, dall'alto in basso (la prima cella verifica o installa il JDK). Nessuna libreria esterna.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `RecursiveSum` → `recursiveSum` (camelCase).

In [1]:
# Verifica (ed eventualmente installa) il JDK
!java -version || apt-get -qq install -y default-jdk-headless

openjdk version "25.0.4.1" 2026-08-18
OpenJDK Runtime Environment (build 25.0.4.1+1-1-26.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 25.0.4.1+1-1-26.04.4-Ubuntu, mixed mode, sharing)


## 1. RecursiveSum e il conteggio dei passi

**RecursiveSum**$(a,n)$ — *Input*: un array $a[1..n]$ di $n\ge1$ interi. *Output*: la somma $a[1]+\dots+a[n]$.

Il codice è fedele allo pseudocodice (righe 2, 3, 5, 6); un contatore `steps` somma i passi eseguiti da ogni chiamata, come nel capitolo: 2 passi se $n=1$, 3 passi più la chiamata ricorsiva se $n>1$. La seconda funzione è `sumRecDecrease` del codice del corso, che si ferma all'array vuoto ($n=0$) e quindi esegue una chiamata in più.

In [2]:
%%writefile RecursiveSum.java
import java.util.*;

public class RecursiveSum {
    static long steps = 0;   // passi eseguiti (righe di pseudocodice)
    static int calls = 0;    // numero di chiamate

    // Somma di a[0..n-1] (pseudocodice: a[1..n]), n >= 1
    static int recursiveSum(int[] a, int n) {
        calls++;
        steps++;                  // riga 2: if n = 1
        if (n == 1) {
            steps++;              // riga 3: return a[1]
            return a[0];
        }
        steps += 2;               // righe 5 e 6
        int s = recursiveSum(a, n - 1);
        return s + a[n - 1];
    }

    // Variante del codice del corso: caso base n = 0
    static long sumRecDecrease(int[] a, int n) {
        calls++;
        if (n == 0) {
            return 0;
        }
        return sumRecDecrease(a, n - 1) + a[n - 1];
    }

    public static void main(String[] args) {
        int[] a = {3, 1, 4, 1, 5, 9, 2, 6};
        int n = 8;
        System.out.println("a = " + Arrays.toString(a));
        calls = 0;
        int r = recursiveSum(a, n);
        System.out.println("RecursiveSum(a, 8) = " + r + " con " + calls + " chiamate");
        calls = 0;
        long r2 = sumRecDecrease(a, n);
        System.out.println("sumRecDecrease(a, 8) = " + r2 + " con " + calls + " chiamate");
        System.out.println("n  S(n) misurato  3n-1");
        for (int k = 1; k <= 8; k++) {
            steps = 0;
            recursiveSum(a, k);
            System.out.printf("%-2d %12d %5d%n", k, steps, 3 * k - 1);
        }
    }
}

Writing RecursiveSum.java


In [3]:
!javac RecursiveSum.java && java RecursiveSum

a = [3, 1, 4, 1, 5, 9, 2, 6]
RecursiveSum(a, 8) = 31 con 8 chiamate
sumRecDecrease(a, 8) = 31 con 9 chiamate
n  S(n) misurato  3n-1
1             2     2
2             5     5
3             8     8
4            11    11
5            14    14
6            17    17
7            20    20
8            23    23


**Prova tu.** Cambia l'array `a` (anche la lunghezza) e verifica che i passi misurati siano sempre $3n-1$.

In [4]:
%%writefile ProvaRecursiveSum.java
import java.util.*;

public class ProvaRecursiveSum {
    public static void main(String[] args) {
        // Cambia l'array e controlla che i passi misurati siano sempre 3n-1.
        int[] a = {7, -2, 5, 0, 11};
        int n = 5;
        RecursiveSum.steps = 0;
        System.out.println(RecursiveSum.recursiveSum(a, n) + " " + RecursiveSum.steps + " " + (3 * n - 1));
    }
}

Writing ProvaRecursiveSum.java


In [5]:
!javac ProvaRecursiveSum.java && java ProvaRecursiveSum

21 14 14


## 2. Torre di Hanoi: $T(n)=2T(n-1)+1$

**Hanoi**$(n, from, to, aux)$ — *Input*: il numero di dischi $n\ge1$ e i tre pioli. *Output*: la sequenza di mosse che trasferisce i dischi $1,\dots,n$ da *from* a *to*.

Il codice conta le mosse (e, per $n=3$, le stampa): il capitolo mostra che sono esattamente $2^n-1$. La chiamata iniziale è `Hanoi(n, 'A', 'C', 'B')`.

In [6]:
%%writefile Hanoi.java
import java.util.*;

public class Hanoi {
    static long moves = 0;    // numero di mosse
    static boolean printMoves = false;

    static void hanoi(int n, char from, char to, char aux) {
        if (n == 1) {
            moves++;                                // sposta il disco 1
            if (printMoves) System.out.println("  disco 1: " + from + " -> " + to);
            return;
        }
        hanoi(n - 1, from, aux, to);
        moves++;                                    // sposta il disco n
        if (printMoves) System.out.println("  disco " + n + ": " + from + " -> " + to);
        hanoi(n - 1, aux, to, from);
    }

    public static void main(String[] args) {
        System.out.println("Mosse per n = 3:");
        printMoves = true;
        hanoi(3, 'A', 'C', 'B');
        printMoves = false;
        System.out.println("n   mosse  2^n-1");
        int[] ns = {1, 2, 3, 4, 5, 6, 7, 8, 10};
        for (int n : ns) {
            moves = 0;
            hanoi(n, 'A', 'C', 'B');
            System.out.printf("%-3d %5d %5d%n", n, moves, (1L << n) - 1);
        }
    }
}

Writing Hanoi.java


In [7]:
!javac Hanoi.java && java Hanoi

Mosse per n = 3:
  disco 1: A -> C
  disco 2: A -> B
  disco 1: C -> B
  disco 3: A -> C
  disco 1: B -> A
  disco 2: B -> C
  disco 1: A -> C
n   mosse  2^n-1
1       1     1
2       3     3
3       7     7
4      15    15
5      31    31
6      63    63
7     127   127
8     255   255
10   1023  1023


**Prova tu.** Cambia il numero di dischi $n$: le mosse crescono come $2^n-1$ (con $n=15$ sono già $32\,767$).

In [8]:
%%writefile ProvaHanoi.java
import java.util.*;

public class ProvaHanoi {
    public static void main(String[] args) {
        // Prova con più dischi: quante mosse servono per n = 15?
        int n = 15;
        Hanoi.moves = 0;
        Hanoi.hanoi(n, 'A', 'C', 'B');
        System.out.println(n + " " + Hanoi.moves + " " + ((1L << n) - 1));
    }
}

Writing ProvaHanoi.java


In [9]:
!javac ProvaHanoi.java && java ProvaHanoi

15 32767 32767


## 3. Ricorrenze calcolate direttamente e forme chiuse

Ogni ricorrenza è calcolata con una funzione ricorsiva che segue la definizione e confrontata con la forma chiusa ottenuta nel capitolo (metodo dell'espansione). Per le ricorrenze con $n/2$ i valori di $n$ sono potenze di 2.

| ricorrenza | forma chiusa |
|---|---|
| $T(n)=T(n-1)+n$, $T(1)=1$ | $n(n+1)/2$ |
| $S(n)=S(n-1)+3$, $S(1)=2$ (RecursiveSum) | $3n-1$ |
| $T(n)=2T(n-1)+1$, $T(1)=1$ (Hanoi) | $2^n-1$ |
| $T(n)=T(n/2)+1$, $T(1)=1$ | $\log_2 n+1$ |
| $T(n)=2T(n/2)+n$, $T(1)=1$ | $n\log_2 n+n$ |

In [10]:
%%writefile Recurrences.java
import java.util.*;

public class Recurrences {
    static long tQuad(long n)  { return n == 1 ? 1 : tQuad(n - 1) + n; }
    static long tSum(long n)   { return n == 1 ? 2 : tSum(n - 1) + 3; }
    static long tHanoi(long n) { return n == 1 ? 1 : 2 * tHanoi(n - 1) + 1; }
    static long tLog(long n)   { return n == 1 ? 1 : tLog(n / 2) + 1; }
    static long tNLogN(long n) { return n == 1 ? 1 : 2 * tNLogN(n / 2) + n; }

    static long log2(long n) {   // n è una potenza di 2
        return 63 - Long.numberOfLeadingZeros(n);
    }

    static void confronta(String nome, long[] valori, java.util.function.LongUnaryOperator ric, java.util.function.LongUnaryOperator chiusa) {
        System.out.println(nome);
        for (long n : valori) {
            long r = ric.applyAsLong(n), c = chiusa.applyAsLong(n);
            System.out.printf("  n=%-7d %12d %12d  %s%n", n, r, c, r == c ? "OK" : "DIVERSI");
        }
    }

    public static void main(String[] args) {
        confronta("T(n) = T(n-1) + n  ->  n(n+1)/2", new long[]{1, 2, 3, 4, 10, 100}, Recurrences::tQuad, n -> n * (n + 1) / 2);
        confronta("S(n) = S(n-1) + 3  ->  3n-1", new long[]{1, 2, 3, 4, 100}, Recurrences::tSum, n -> 3 * n - 1);
        confronta("T(n) = 2T(n-1) + 1  ->  2^n-1", new long[]{1, 2, 3, 4, 10, 20}, Recurrences::tHanoi, n -> (1L << n) - 1);
        confronta("T(n) = T(n/2) + 1  ->  log2(n)+1", new long[]{1, 2, 4, 8, 1024, 1 << 20}, Recurrences::tLog, n -> log2(n) + 1);
        confronta("T(n) = 2T(n/2) + n  ->  n log2(n) + n", new long[]{1, 2, 4, 8, 1024, 1 << 20}, Recurrences::tNLogN, n -> n * log2(n) + n);
    }
}

Writing Recurrences.java


In [11]:
!javac Recurrences.java && java Recurrences

T(n) = T(n-1) + n  ->  n(n+1)/2
  n=1                  1            1  OK
  n=2                  3            3  OK
  n=3                  6            6  OK
  n=4                 10           10  OK
  n=10                55           55  OK
  n=100             5050         5050  OK
S(n) = S(n-1) + 3  ->  3n-1
  n=1                  2            2  OK
  n=2                  5            5  OK
  n=3                  8            8  OK
  n=4                 11           11  OK
  n=100              299          299  OK
T(n) = 2T(n-1) + 1  ->  2^n-1
  n=1                  1            1  OK
  n=2                  3            3  OK
  n=3                  7            7  OK
  n=4                 15           15  OK
  n=10              1023         1023  OK
  n=20           1048575      1048575  OK
T(n) = T(n/2) + 1  ->  log2(n)+1
  n=1                  1            1  OK
  n=2                  2            2  OK
  n=4                  3            3  OK
  n=8                  4            4

**Prova tu.** Aggiungi una ricorrenza tua (qui $T(n)=T(n-1)+2n-1$, $T(1)=1$) e confrontala con la sua forma chiusa.

In [12]:
%%writefile ProvaRecurrences.java
import java.util.*;

public class ProvaRecurrences {
    static long tMia(long n) { return n == 1 ? 1 : tMia(n - 1) + 2 * n - 1; }

    public static void main(String[] args) {
        // Aggiungi una ricorrenza: T(n) = T(n-1) + 2n - 1 con T(1) = 1 ha forma chiusa n^2.
        Recurrences.confronta("T(n) = T(n-1) + 2n - 1  ->  n^2", new long[]{1, 2, 3, 10, 100}, ProvaRecurrences::tMia, n -> n * n);
    }
}

Writing ProvaRecurrences.java


In [13]:
!javac ProvaRecurrences.java && java ProvaRecurrences

T(n) = T(n-1) + 2n - 1  ->  n^2
  n=1                  1            1  OK
  n=2                  4            4  OK
  n=3                  9            9  OK
  n=10               100          100  OK
  n=100            10000        10000  OK
